<a href="https://colab.research.google.com/github/Gayathri-rfr/DataPOC/blob/main/api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import numpy as np

# Absolute local import of our DB layout
import database

# Initialize database on API startup
database.init_db()

app = FastAPI(title="Data Ingestion Gateway")

class Transaction(BaseModel):
    tx_id: str
    user_id: int
    amount: float
    category: str

@app.post("/ingest/")
async def ingest_transaction(tx: Transaction):
    # NumPy: Vectorized baseline calculations to identify statistical outliers
    baseline_amounts = np.array([10.0, 25.5, 100.0, 50.0, tx.amount])
    mean = np.mean(baseline_amounts)
    std = np.std(baseline_amounts)

    # Calculate Z-score safely
    z_score = (tx.amount - mean) / std if std > 0 else 0
    is_anomaly = bool(z_score > 1.5)

    # Prepare data payload for Django mock storage
    payload = tx.model_dump()
    payload["is_anomaly"] = is_anomaly

    saved = database.save_to_django_db(payload)

    if not saved:
        raise HTTPException(status_code=400, detail="Transaction ID already exists.")

    return {
        "status": "Processed",
        "anomaly_flag": is_anomaly,
        "saved_to_django_records": saved
    }


